# Environment Setup & Path Selection — Week 5

**Learning Objectives:**
- Understand the three fine-tuning paths (MLX, HF+TRL, Cloud GPU) and select yours
- Inspect the base model tokenizer to understand vocabulary and chat templates
- Run a baseline inference on the un-tuned model so you have a before/after reference

**Estimated Time:** 15 minutes

**Path Indicator:** All paths start here. Path A (MLX) and Path B (HF+TRL) diverge in later notebooks.

In [1]:
import sys
import importlib

sys.path.insert(0, "..")

import src
importlib.reload(src)

from dotenv import load_dotenv
load_dotenv(override=True)

%matplotlib inline

from src.cost_tracker import CostTracker
from src.llm_client import LLMClient
from src.config import (
    PATH, CLAUDE_MODEL, OLLAMA_MODEL,
    BASE_MODEL_HF, BASE_MODEL_MLX, FINETUNE_BACKEND
)

tracker = CostTracker()
# Path C = hybrid: can use both Claude and Ollama
llm = LLMClient(path="C")

print("Imports OK")
print(f"  PATH              = {PATH}")
print(f"  FINETUNE_BACKEND  = {FINETUNE_BACKEND}")
print(f"  BASE_MODEL_HF     = {BASE_MODEL_HF}")
print(f"  BASE_MODEL_MLX    = {BASE_MODEL_MLX}")

✓ Claude API client initialized
  Default model: claude-sonnet-4-6
  Available: claude-sonnet-4-6, claude-opus-4-6, claude-haiku-4-5-20251001
✓ Ollama client initialized
  Available models: ['llama3.2-vision:latest', 'qwen3.5:27b', 'qwen3.5:9b']
  Default model: qwen3.5:27b
Imports OK
  PATH              = A
  FINETUNE_BACKEND  = mlx
  BASE_MODEL_HF     = Qwen/Qwen2.5-0.5B-Instruct
  BASE_MODEL_MLX    = mlx-community/Qwen2.5-0.5B-Instruct-bf16


## Part 1: Path Selection

There are three fine-tuning paths available this week. Choose based on your hardware.

| Path | Backend | Hardware | Speed (0.5B) | Mirrors Lecture? |
|------|---------|----------|-------------|------------------|
| **A** | MLX-LM | Apple Silicon Mac only | 5–15 min | Partial |
| **B** | HF + TRL + QLoRA | Any GPU (8–16 GB VRAM) or CPU | 10–30 min | Yes |
| **C** | Cloud GPU (Colab Pro / RunPod) | Any machine, cloud GPU | Any model size | Yes (bonus) |

**Path A** uses Apple's MLX framework — it's the fastest option on M1/M2/M3 Macs and requires no separate CUDA setup. The tradeoff is that MLX-LM's API is slightly different from the HuggingFace ecosystem covered in lecture.

**Path B** mirrors the lecture exactly (PEFT + TRL SFTTrainer + QLoRA). If you have a discrete GPU with ≥8 GB VRAM (e.g., an RTX 3080), this is the recommended path.

**Path C** is a bonus path — use a cloud GPU notebook (Colab Pro, RunPod, Lambda Labs) and you can train larger models.

To override the auto-detected backend, add this line to your `.env`:
```
FINETUNE_BACKEND=mlx   # or: hf
```

In [2]:
import platform
import os

print(f"Detected system : {platform.system()} {platform.machine()}")
print(f"FINETUNE_BACKEND: {FINETUNE_BACKEND}")
print()

if FINETUNE_BACKEND == "mlx":
    print("You are on PATH A (MLX-LM). Subsequent notebooks will use mlx_lm.")
    print(f"Base model: {BASE_MODEL_MLX}")
elif FINETUNE_BACKEND == "hf":
    print("You are on PATH B (HF + TRL). Subsequent notebooks will use transformers + trl.")
    print(f"Base model: {BASE_MODEL_HF}")
else:
    print(f"Unknown backend: {FINETUNE_BACKEND} — check your .env or config.py")

print()
env_override = os.environ.get("FINETUNE_BACKEND", "(not set — auto-detected)")
print(f"FINETUNE_BACKEND env var: {env_override}")

Detected system : Darwin arm64
FINETUNE_BACKEND: mlx

You are on PATH A (MLX-LM). Subsequent notebooks will use mlx_lm.
Base model: mlx-community/Qwen2.5-0.5B-Instruct-bf16

FINETUNE_BACKEND env var: mlx


## Part 2: Model Download Check

We load **only the tokenizer** for `Qwen/Qwen2.5-0.5B-Instruct` — this is ~3 MB versus ~1 GB for the full model weights. This confirms HuggingFace connectivity and lets us inspect the vocabulary before we commit to a full download.

In [3]:
print(f"Loading tokenizer for: {BASE_MODEL_HF}")
print("(This downloads only the tokenizer config — ~3 MB)")
print()

try:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_HF)

    print(f"Vocab size       : {tokenizer.vocab_size:,}")
    print(f"Model max length : {tokenizer.model_max_length}")
    print(f"Tokenizer class  : {type(tokenizer).__name__}")

    chat_template = getattr(tokenizer, 'chat_template', None)
    if chat_template:
        print(f"Chat template    : present ({len(chat_template)} chars)")
        # Show the first 200 chars of the template
        print(f"  Preview: {chat_template[:200]}...")
    else:
        print("Chat template    : not set")

    # Show special tokens
    print()
    print("Special tokens:")
    print(f"  BOS: {tokenizer.bos_token!r} (id={tokenizer.bos_token_id})")
    print(f"  EOS: {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")
    print(f"  PAD: {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")

except Exception as e:
    print(f"Error loading tokenizer: {e}")
    print("Check your internet connection or HuggingFace token if the model is gated.")

Loading tokenizer for: Qwen/Qwen2.5-0.5B-Instruct
(This downloads only the tokenizer config — ~3 MB)



config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab size       : 151,643
Model max length : 131072
Tokenizer class  : Qwen2Tokenizer
Chat template    : present (2507 chars)
  Preview: {%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Clou...

Special tokens:
  BOS: None (id=None)
  EOS: '<|im_end|>' (id=151645)
  PAD: '<|endoftext|>' (id=151643)


## Part 3: First Inference on Un-Tuned Base

Before we fine-tune anything, let's record what the base model says about itself. We use Ollama (running `qwen3.5:27b` as our local proxy) to answer this question. After fine-tuning on a resume in later notebooks, you'll be able to compare the model's behavior before and after.

**Why this matters:** Fine-tuning changes the model's _style, tone, and factual grounding_ on a specific domain. Having a baseline is essential for measuring improvement.

In [4]:
baseline_prompt = "What is your name and what are you trained on?"

print(f"Prompt: {baseline_prompt!r}")
print()
print("--- Ollama response (baseline) ---")

try:
    resp = llm.generate(
        prompt=baseline_prompt,
        model=OLLAMA_MODEL,
        max_tokens=256,
        use_claude=False
    )
    tracker.add_call(resp)
    baseline_response = resp.get("content", resp.get("error"))
    print(baseline_response)
except Exception as e:
    baseline_response = None
    print(f"Ollama error: {e}")
    print("Make sure `ollama serve` is running and the model is pulled.")

Prompt: 'What is your name and what are you trained on?'

--- Ollama response (baseline) ---
My name is **Qwen3.5**, and I am a large language model developed by Tongyi Lab. I am trained on a vast and diverse dataset of text from the internet, books, articles, and other publicly available sources, with knowledge cutoff in **2026**. This training enables me to understand and generate human-like text across a wide range of topics, though I do not have access to real-time data or private information. How can I assist you today? 😊


### TODO 1

Change the system prompt below and rerun the cell. What changed in the response? Try something like:
- `"You are a pirate. Answer all questions in pirate speak."`
- `"You are a concise assistant. Answer in exactly one sentence."`

Describe what you observe.

In [5]:
# TODO 1: Change the system prompt and observe the effect

custom_system = "You are a helpful assistant."
# ^ CHANGE THIS

try:
    resp = llm.generate(
        prompt=baseline_prompt,
        system=custom_system,
        model=OLLAMA_MODEL,
        max_tokens=256,
        use_claude=False
    )
    tracker.add_call(resp)
    custom_response = resp.get("content", resp.get("error"))
    print(f"System: {custom_system!r}")
    print(f"Response:\n{custom_response}")
except Exception as e:
    print(f"Error: {e}")

System: 'You are a helpful assistant.'
Response:
Hello! I'm **Qwen3.5**, the latest large language model developed by Tongyi Lab. My training data is sourced from **high-quality, diverse internal data within Alibaba Group**, and my knowledge cutoff is **2026**. This includes vast amounts of text, code, and multilingual content, enabling me to assist with complex tasks across domains like reasoning, coding, scientific analysis, and creative work. How can I help you today? 😊


In [ ]:
# TODO 1 reflection -- edit your answer below, then run this cell.
todo1_reflection = """[Replace with your answer]

Hint: Compare the response from the default system prompt vs. your custom one (e.g. the pirate or one-sentence-only prompt). In 2-3 sentences, describe what changed in tone, length, or content, and explain why a system prompt has this effect on a chat-tuned model.
"""
print(todo1_reflection)


### TODO 2

In 2–3 sentences, describe the difference between a **base model** and an **instruct model**:
- What kind of data is each trained on?
- When would you use each?
- Why does `Qwen2.5-0.5B-Instruct` respond to questions but `Qwen2.5-0.5B` (base) might not?

In [ ]:
# TODO 2 reflection -- edit your answer below, then run this cell.
todo2_reflection = """[Replace with your answer]

Hint: In 2-3 sentences, contrast a base model vs. an instruct model. Cover (a) what data each is trained on (raw web text vs. SFT/RLHF on instruction pairs), (b) when you would pick each as a starting point, and (c) why Qwen2.5-0.5B-Instruct can answer questions while the base Qwen2.5-0.5B might just continue the prompt.
"""
print(todo2_reflection)


## Summary

In [ ]:
from src.utils import append_to_reflection

# Build reflection from student TODO answers (auto-captured)
section_text = (
    "### TODO 1\n" + (todo1_reflection if 'todo1_reflection' in dir() else "[not completed]") + "\n\n" +
    "### TODO 2\n" + (todo2_reflection if 'todo2_reflection' in dir() else "[not completed]")
)
append_to_reflection(
    notebook="01",
    section_title="Environment Setup & Path Selection",
    reflection_content=section_text,
)
print("Reflection auto-saved to outputs/homework_reflection.md")
tracker.report()
